# Lecture 4 — Class Exercise
## Scatter & Bubble Charts: Gapminder

> **Push to:** `week04/lecture04_exercise.ipynb`

**Rules:**
1. Colour used **sparingly** — one categorical variable, no rainbow
2. If showing all continents, either use accessible palette OR grey all + highlight one
3. `size_max` set when using bubble size
4. Log scale for GDP per capita
5. Insight title

---


In [ ]:
import pandas as pd
import plotly.express as px


# Dataset: Gapminder — GDP, Life Expectancy, Population by Country
# Source: Gapminder Foundation (gapminder.org)

df = px.data.gapminder()
print(f"Loaded: {len(df)} rows")
print(df.head())

In [ ]:
# explore

print(df.info())
print("Years:", sorted(df['year'].unique()))
print("Continents:", df['continent'].unique())
print(df.describe().round(1))


## Task 1 — Scatter: life expectancy change over time

**What to build:** A scatter showing **GDP per capita vs life expectancy** for **two years** (2002 and 2007) to show how both moved — use **colour for year** (just 2 colours), **one continent only**.

Choose any continent except Africa (that was the example). Highlight the change direction.

> 💡 Filter: `df.loc[df['continent'] == 'YOUR_CHOICE']` then filter years


In [ ]:
# Task 1 — Asia, 2002 vs 2007: GDP vs life expectancy
# Colour encodes year (two colours only). 2007 is the stronger colour so the shift stands out.

import pandas as pd
import plotly.express as px

df = px.data.gapminder()

asia = df.loc[
    (df['continent'] == 'Asia') & (df['year'].isin([2002, 2007]))
].copy()
asia['year'] = asia['year'].astype(str)  # treat year as a category, not a continuous scale

fig = px.scatter(
    data_frame=asia,
    x='gdpPercap',
    y='lifeExp',
    color='year',
    color_discrete_map={'2002': '#AAB7B8', '2007': 'crimson'},
    hover_name='country',
    hover_data={'year': True, 'continent': False},
    log_x=True,
    labels={
        'gdpPercap': 'GDP per Capita (USD, log scale)',
        'lifeExp': 'Life Expectancy',
        'year': '',
    },
    template='plotly_white',
    title='Asian countries moved up and right from 2002 to 2007 — longer lives and higher GDP, but Afghanistan stayed behind',
)

fig.update_traces(marker=dict(size=9, opacity=0.8))
fig.update_traces(selector=dict(name='2007'), marker=dict(size=11, opacity=0.9))

fig.add_annotation(
    x=2.0, y=45,
    text='2002 → 2007:\nup and right',
    showarrow=False,
    font=dict(color='crimson', size=12, family='Arial'),
    bgcolor='white',
    align='left',
)

fig.update_layout(
    font=dict(family='Arial', size=13),
    height=500,
    legend=dict(orientation='h', x=0, y=1.08, title=''),
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
)

fig.show()


## Task 2 — Bubble chart: tell a story

**What to build:** A bubble chart (full 2007 dataset, all countries) where:
- x = GDP per capita (log scale)
- y = life expectancy
- size = population
- colour = ONE continent highlighted (your choice), all others grey
- At least one annotation explaining the highlighted group's story

> This is the grey-and-highlight technique applied to a bubble chart.


In [ ]:
# Task 2 — 2007 bubble chart: grey every continent, highlight Asia

import math
import pandas as pd
import plotly.express as px

df = px.data.gapminder(year=2007)

highlight_continent = 'Asia'
colour_map = {
    c: ('crimson' if c == highlight_continent else 'lightgrey')
    for c in df['continent'].unique()
}

fig = px.scatter(
    data_frame=df,
    x='gdpPercap',
    y='lifeExp',
    size='pop',
    size_max=60,
    color='continent',
    color_discrete_map=colour_map,
    hover_name='country',
    hover_data={'continent': False},
    log_x=True,
    labels={
        'gdpPercap': 'GDP per Capita (log scale)',
        'lifeExp': 'Life Expectancy',
        'pop': 'Population',
        'continent': '',
    },
    template='plotly_white',
    title='Asia is not one cluster — Japan sits with the West while Afghanistan remains at African levels of wealth and health',
    opacity=0.75,
)

fig.update_traces(
    selector=dict(name=highlight_continent),
    marker=dict(opacity=0.85),
)
fig.update_traces(
    selector=lambda t: t.name != highlight_continent,
    marker=dict(opacity=0.45),
)

# log_x means annotation x must be log10 of GDP (same trick as the lecture notes)
for country, ax, ay in [('Japan', 50, -25), ('Afghanistan', 55, 35)]:
    row = df[df['country'] == country].iloc[0]
    fig.add_annotation(
        x=math.log10(row['gdpPercap']),
        y=row['lifeExp'],
        text=country,
        showarrow=True,
        arrowcolor='crimson',
        arrowhead=1,
        ax=ax,
        ay=ay,
        font=dict(size=12, family='Arial', color='crimson'),
    )

fig.add_annotation(
    x=math.log10(1500),
    y=48,
    text='Asia spans the full\nwealth–health range',
    showarrow=False,
    font=dict(color='crimson', size=11, family='Arial'),
    bgcolor='white',
    bordercolor='crimson',
    borderwidth=1,
    borderpad=4,
)

fig.update_layout(
    font=dict(family='Arial', size=13),
    showlegend=False,
    height=550,
    yaxis=dict(showgrid=False),
    xaxis=dict(showgrid=True, gridcolor='#EEEEEE'),
)

fig.show()
